# Neighbourhood Types in Essen – Clustering with Census 2022 (Zensus 2022)

Multivariate clustering (k-means) of the 50 districts (*Stadtteile*) of Essen, based on 100 m grid data of the German Census 2022.

**Steps:** district boundaries (OpenStreetMap) → census grid data (Destatis) → aggregation to districts → standardisation → k-means (k chosen by silhouette score) → maps and profiles.

In [ ]:
# 1. Install packages
!pip install -q osmnx contextily

In [ ]:
# 2. Imports and settings
import io, zipfile, requests, warnings
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox, contextily as ctx
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
warnings.filterwarnings("ignore")
import os
os.makedirs("images", exist_ok=True); os.makedirs("results", exist_ok=True)

BASE = "https://www.destatis.de/static/DE/zensus/gitterdaten/"
# variable name : (zip file, label for maps)
VARS = {
    "einwohner":   ("Zensus2022_Bevoelkerungszahl.zip", "Einwohner"),
    "alter":       ("Durchschnittsalter_in_Gitterzellen.zip", "Durchschnittsalter"),
    "u18":         ("Anteil_unter_18-jaehrige_in_Gitterzellen.zip", "Anteil unter 18 (%)"),
    "ue65":        ("Anteil_ab_65-jaehrige_in_Gitterzellen.zip", "Anteil ab 65 (%)"),
    "auslaender":  ("Auslaenderanteil_in_Gitterzellen.zip", "Ausländeranteil (%)"),
    "hhgroesse":   ("Durchschnittliche_Haushaltsgroesse_in_Gitterzellen.zip", "Haushaltsgröße"),
    "miete":       ("Zensus2022_Durchschn_Nettokaltmiete.zip", "Nettokaltmiete (€/m²)"),
    "eigentum":    ("Eigentuemerquote_in_Gitterzellen.zip", "Eigentümerquote (%)"),
    "leerstand":   ("Leerstandsquote_in_Gitterzellen.zip", "Leerstandsquote (%)"),
    "wohnflaeche": ("Durchschnittliche_Wohnflaeche_je_Bewohner_in_Gitterzellen.zip", "Wohnfläche je Bewohner (m²)"),
}

In [ ]:
# 3. District boundaries (Stadtteile, admin_level 10) from OpenStreetMap
st = ox.features_from_place("Essen, Germany", tags={"boundary": "administrative", "admin_level": "10"})
st = st[st.geometry.type.isin(["Polygon", "MultiPolygon"])][["name", "geometry"]].reset_index(drop=True)
stadt = ox.geocode_to_gdf("Essen, Germany").to_crs(3035)
st = st.to_crs(3035)
st = st[st.representative_point().within(stadt.geometry.iloc[0].buffer(50))].reset_index(drop=True)
st = st[(st["name"] != "Essen") & (~st["name"].str.startswith("Stadtbezirk"))].reset_index(drop=True)
st["flaeche_km2"] = st.area / 1e6
print(len(st), "Stadtteile")
st.plot(edgecolor="black", facecolor="none", figsize=(6, 6));

In [ ]:
# 4. Download census grid data (100 m) and keep only cells in Essen
xmin, ymin, xmax, ymax = st.total_bounds

def load_var(zipname):
    r = requests.get(BASE + zipname, timeout=600); r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    names = [n for n in z.namelist() if n.lower().endswith(".csv")]
    name = [n for n in names if "100m" in n.lower()][0]
    raw = z.read(name)
    enc = "utf-8"
    try: raw[:200000].decode("utf-8")
    except UnicodeDecodeError: enc = "latin1"
    parts = []
    for ch in pd.read_csv(io.BytesIO(raw), sep=";", decimal=",", encoding=enc, chunksize=500000, low_memory=False):
        cols = {c.lower(): c for c in ch.columns}
        xc = [cols[c] for c in cols if c.startswith("x_mp")][0]
        yc = [cols[c] for c in cols if c.startswith("y_mp")][0]
        ic = [cols[c] for c in cols if c.startswith("gitter_id")][0]
        ch = ch[(ch[xc] >= xmin) & (ch[xc] <= xmax) & (ch[yc] >= ymin) & (ch[yc] <= ymax)]
        parts.append(ch)
    df = pd.concat(parts)
    val = [c for c in df.columns if c not in (ic, xc, yc) and "werterlaeuternde" not in c.lower()][0]
    out = df[[ic, xc, yc, val]].copy()
    out.columns = ["id", "x", "y", "wert"]
    out["wert"] = pd.to_numeric(out["wert"].astype(str).str.replace(",", "."), errors="coerce")
    print(f"{zipname}: {len(out)} cells, column '{val}'")
    return out

grid = None
for key, (zn, _) in VARS.items():
    d = load_var(zn).rename(columns={"wert": key})
    grid = d if grid is None else grid.merge(d[["id", key]], on="id", how="left")
grid = grid[grid["einwohner"] > 0]
print(len(grid), "populated 100 m cells")

In [ ]:
# 5. Assign cells to districts and aggregate (population-weighted means)
pts = gpd.GeoDataFrame(grid, geometry=gpd.points_from_xy(grid.x, grid.y), crs=3035)
pts = gpd.sjoin(pts, st[["name", "geometry"]], predicate="within").drop(columns="index_right")

def wmean(g, col):
    m = g[col].notna()
    return np.average(g.loc[m, col], weights=g.loc[m, "einwohner"]) if m.any() else np.nan

rows = []
for name, g in pts.groupby("name"):
    r = {"name": name, "einwohner": g["einwohner"].sum()}
    for k in VARS:
        if k != "einwohner": r[k] = wmean(g, k)
    rows.append(r)
agg = st.merge(pd.DataFrame(rows), on="name", how="inner")
agg["dichte"] = agg["einwohner"] / agg["flaeche_km2"]
print("Einwohner gesamt:", int(agg["einwohner"].sum()))
agg.drop(columns="geometry").round(1).head()

In [ ]:
# 6. Standardise and run k-means; choose k by silhouette score
FEAT = ["dichte", "alter", "u18", "ue65", "auslaender", "hhgroesse", "miete", "eigentum", "leerstand", "wohnflaeche"]
X = agg[FEAT].copy()
X["dichte"] = np.log(X["dichte"])
X = X.fillna(X.median())
Z = StandardScaler().fit_transform(X)

scores = {}
for k in range(3, 8):
    lab = KMeans(n_clusters=k, n_init=50, random_state=42).fit_predict(Z)
    scores[k] = silhouette_score(Z, lab)
print({k: round(v, 3) for k, v in scores.items()})
K = max(scores, key=scores.get)
km = KMeans(n_clusters=K, n_init=50, random_state=42).fit(Z)

# order clusters by average rent (1 = lowest rent)
agg["c"] = km.labels_
order = agg.groupby("c")["miete"].mean().sort_values().index
agg["cluster"] = agg["c"].map({c: i + 1 for i, c in enumerate(order)})
agg = agg.drop(columns="c")
print("k =", K)
agg.groupby("cluster")[FEAT + ["einwohner"]].mean().round(1)

In [ ]:
# 7. Cluster profiles (mean z-score per variable)
zdf = pd.DataFrame(Z, columns=FEAT); zdf["cluster"] = agg["cluster"].values
prof = zdf.groupby("cluster").mean()
labels = ["Einwohnerdichte (log)"] + [VARS[f][1] for f in FEAT[1:]]
fig, ax = plt.subplots(figsize=(11, 0.9 * K + 2))
im = ax.imshow(prof.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(FEAT))); ax.set_xticklabels(labels, rotation=40, ha="right")
ax.set_yticks(range(K)); ax.set_yticklabels([f"Typ {i}" for i in prof.index])
for i in range(K):
    for j in range(len(FEAT)):
        ax.text(j, i, f"{prof.values[i, j]:.1f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, label="z-Wert (Abweichung vom Essener Mittel)")
ax.set_title("Profil der Stadtteil-Typen – Essen, Zensus 2022")
plt.tight_layout(); plt.savefig("images/02_Cluster_Profile.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# 8. Map of district types
COLORS = ["#d7191c", "#fdae61", "#abd9e9", "#2c7bb6", "#1a9641", "#984ea3", "#636363"]
m = agg.to_crs(3857)
fig, ax = plt.subplots(figsize=(10, 10))
for c in sorted(m["cluster"].unique()):
    m[m.cluster == c].plot(ax=ax, color=COLORS[c - 1], edgecolor="white", linewidth=0.8, alpha=0.85, label=f"Typ {c}")
for _, r in m.iterrows():
    p = r.geometry.representative_point()
    ax.annotate(r["name"], (p.x, p.y), ha="center", fontsize=6)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas, attribution_size=6)
ax.legend(title="Stadtteil-Typ", loc="lower left")
ax.set_title(f"Stadtteil-Typen in Essen – k-means-Clustering (k = {K}), Zensus 2022")
ax.set_axis_off()
plt.savefig("images/01_Stadtteil_Typen_Essen.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# 9. Export results and download
out = agg.copy()
out.to_crs(25832).to_file("results/stadtteil_typen_essen.gpkg", driver="GPKG")
out.drop(columns="geometry").round(2).sort_values(["cluster", "name"]).to_csv("results/stadtteil_typen_essen.csv", index=False)
pd.DataFrame({"k": list(scores), "silhouette": [round(v, 3) for v in scores.values()]}).to_csv("results/silhouette_scores.csv", index=False)
!zip -q -r essen_clustering_results.zip images results
from google.colab import files
files.download("essen_clustering_results.zip")